# POC 05c — client similarity and clustering: raw vs reconstructed prototypes

Does the similarity between clients, read from the prototypes the protocol exchanges, reproduce
the similarity in the data, recover the regime grouping, and follow the drift? Asked of the **raw
latent** prototypes and of the **reconstructed** (decoded) ones, on every stored run, and scored
against two centralized ceilings.

Same machinery as POC_05b: nothing trains (`assert_cached` + cache-hit check), a failing stage is
recorded and the sweep moves on, every run is checkpointed, and the output is one zip —
`POC_05c_similarity_bundle.zip` — to send back.

**Scope limits, known before running.** Only `f335b2cfad67` has a non-trivial regime partition in
*both* init and final. In the homogeneous worlds init is one group (nothing to cluster), `ece8…`
has a 1-month final phase and `16ce…` none; those worlds answer the tracking question. World
`c5c8…` has no world files: raw latent similarity only.

## 0. Setup

In [1]:
import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"   # BEFORE numpy
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))

import sys, pathlib, json, time, threading, warnings
import numpy as np, pandas as pd

REPO = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "src" / "fedwater").is_dir()), None)
LAB_DIR = (REPO / "notebooks" / "fedwater_lab_refactor") if REPO else pathlib.Path.cwd()
for p in ([REPO / "src"] if REPO else []) + [LAB_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from fedwater_lab import grid as GD, simsweep as SS
from fedwater_lab.checks import openmp_runtimes

try:
    import torch
    torch.get_num_threads()
    _ = torch.ones(256, 256) @ torch.ones(256, 256)
except ImportError:
    torch = None
_ = np.ones((256, 256)) @ np.ones((256, 256))
RUNTIMES = openmp_runtimes()
HAVE_REPO = REPO is not None
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
print("repo:", REPO, "| lab:", LAB_DIR)

repo: c:\Users\arthu\USPy\10_Mestrado\fedWater | lab: c:\Users\arthu\USPy\10_Mestrado\fedWater\notebooks\fedwater_lab_refactor


## 1. Parameters

Same calendar key as POC_05 (weekly period, daily bins, 3-month blocks), so every number here
lines up with the dependence bundle. Cost: one `analyse` per run plus the similarity loops —
about 10–20 s per run, **~10 min for the grid**.

In [2]:
OVERWRITE = False            # True: recompute runs already in parts/
SELECT    = None             # None = every run; or a list of run_keys

P = SS.SimParams(block_months=3, bin_h=24, period_h=168,
                 arms=("pure", "mixed", "all"), cluster_phases=("init", "final"))

STORE_ROOT = LAB_DIR / "lab_sandbox"
RUNS_DIR   = STORE_ROOT / "runs"
OUT_DIR    = LAB_DIR / "poc05c_similarity"
print(P); print("out:", OUT_DIR)

SimParams(block_months=3, bin_h=24, period_h=168, arms=('pure', 'mixed', 'all'), cluster_phases=('init', 'final'), n_perm=19)
out: c:\Users\arthu\USPy\10_Mestrado\fedWater\notebooks\fedwater_lab_refactor\poc05c_similarity


## 2. What is measured

For each calendar group $g=(\text{block},\text{cycle class})$ and client pair $(c,d)$, with
$r(\cdot,\cdot)$ the population Pearson correlation over the window's time steps (`recon._r`, as in
POC_04) and channels matched by slot index:

| space | definition | role |
|---|---|---|
| `s_lat` | $\cos(p_{c,g},\,p_{d,g})$ on the raw latent prototype | federated, raw |
| `s_dec` | $\frac{1}{|I|}\sum_{i\in I} r(\hat x_{c,g,i},\,\hat x_{d,g,i})$, $\hat x=\mathrm{dec}(p)$ | federated, reconstructed |
| `s_dec_abs` | the same with $|r|$ per channel — sign-free, for unoriented runs | federated, reconstructed |
| `s_true`, `s_true_abs` | the same on the mean true windows | ceiling, sensor level (includes mixing) |
| `s_dem` | $r(q_{c,g},\,q_{d,g})$ on each district's own demand at the same window starts | ceiling, demand level |

$I$ is the arm's channel set: `pure`, `mixed` or `all`. `s_lat` and `s_dem` do not depend on the
arm. Ceilings need the raw data and are evaluation-only.

**Q1 — fidelity.** Per phase, a federated space $b$ against a ceiling $a$ over (i) every class-level
row and (ii) the six phase-averaged pair values (what a clustering step consumes): Pearson,
Spearman, and $\text{concordant}=\text{share of pair couples }(i,j),\,a_i\neq a_j,\text{ with }
(a_i-a_j)(b_i-b_j)>0$. The row `s_true` vs `s_dem` is the mixing reference: how much sensor-level
similarity is demand similarity.

**Q2 — clustering against regimes.** Only where `World.regimes(phase)` is defined (init, final).
Average linkage on $D=1-S$ (phase-averaged pair similarity), cut at the true number of regime groups
$k$. `ari` = adjusted Rand index against the regime partition; `top_ok` = the most similar pair
shares a regime; `auc_same` $=P(s_{\text{same}}>s_{\text{different}})$, ties ½. A partition with
$k=1$ or $k=n$ is trivial and not scored. Six pairs per run ⇒ read across runs, not per run.

**Q3 — tracking the drift.** Per block $b$, pair similarity $s_b$ against the drift district's
switched fraction $\pi_b$ (`World.drift_progress`, averaged over the block's months):
$\rho=\mathrm{Spearman}(s_b,\pi_b)$. Expected sign for a pair $(A,X)$, $A$ the drift district:
$-1$ if $X$ shares $A$'s init regime only (A moves away), $+1$ if $X$ shares $A$'s final regime only
(A converges), else 0; every pair without $A$ expects 0. `sign_ok` is scored where the expected sign
is non-zero. The Spearman p over ~24 autocorrelated blocks is optimistic and stored only for
reference.

## 3. The index

In [3]:
store = load_world = None
if HAVE_REPO:
    from fedwater_lab.store import Store
    from fedwater_lab.worlds import load_world
    store = Store(STORE_ROOT)
    IDX = GD.scan(store.root / "runs")
    IDX["store_readable"] = [GD.locate(store, r.world_hash, r.run_key)["exists"]
                             for r in IDX.itertuples(index=False)]
else:
    IDX = GD.scan(RUNS_DIR)
    IDX["store_readable"] = False
IDX = GD.comparable_blocks(IDX)
if SELECT is not None:
    IDX = IDX[IDX["run_key"].isin(SELECT)].reset_index(drop=True)
print(f"{len(IDX)} runs | {IDX['world_hash'].nunique()} worlds | "
      f"store-readable: {int(IDX['store_readable'].sum())}")
display(IDX.groupby(["world_hash", "schedule"]).size().rename("runs").unstack(fill_value=0))

42 runs | 4 worlds | store-readable: 42


schedule,all,commissioning,streaming
world_hash,,,
16ce72c589df,2,2,2
c5c8904c3d8c,2,0,0
ece8f4ad92e5,14,2,2
f335b2cfad67,6,5,5


## 4. The sweep

In [4]:
from tqdm.auto import tqdm

bar = tqdm(total=len(IDX), desc="similarity", bar_format="{desc} {n}/{total} |{bar}| {elapsed}")
_stop = threading.Event()
def _tick():
    while not _stop.wait(1.0):
        bar.refresh()
threading.Thread(target=_tick, daemon=True).start()

_first = {}
def progress(i, n, label, secs):
    if i and "t" not in _first and secs > 0:
        _first["t"] = secs / i
        print(f"~{_first['t']:.0f} s per run -> about {_first['t'] * (n - i) / 60:.0f} min left")
    bar.n = i
    bar.set_description_str(f"similarity · {str(label)[-48:]}")
    bar.refresh()

t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    RESULT = SS.sweep(IDX, P, OUT_DIR, store=store, load_world=load_world,
                      decoded=HAVE_REPO, overwrite=OVERWRITE, progress=progress)
_stop.set(); bar.close()
print(f"sweep finished in {(time.time() - t0) / 60:.1f} min")
display(RESULT)

similarity 0/42 |          | 00:00

~33 s per run -> about 23 min left
sweep finished in 8.5 min


,run_key,seconds,n_failed_stages
0,2f3aba5dcac9,32.9,0
1,57aa26c2408f,13.8,0
2,694733f16841,16.8,0
3,8241f2e9ba64,13.3,0
4,d11b40a20d21,16.3,0
5,da72c9bbae27,15.2,0
6,1dced6f2762d,0.3,0
7,fe4d4a60301a,0.2,0
8,028ca9bf44b3,15.6,0
9,0f14e551356d,14.7,0


## 5. The bundle — send the zip back

In [5]:
BUNDLE = SS.collect(OUT_DIR, IDX, P, extra={
    "openmp_runtimes": RUNTIMES[["user_api", "internal_api", "prefix", "threading_layer",
                                 "filepath"]].astype(str).to_dict("records"),
    "have_repo": HAVE_REPO})
cfg = BUNDLE["configs"]
print(f"runs done: {len(cfg)} | runs with a failed stage: {int((cfg['n_failed_stages'] > 0).sum())}")
err_cols = [c for c in cfg.columns if c.startswith("err:")]
if err_cols:
    bad = cfg.loc[cfg[err_cols].notna().any(axis=1), ["run_key", "label"] + err_cols]
    if len(bad):
        display(bad)
display(pd.Series(BUNDLE["manifest"]["tables"], name="rows").to_frame())
print("\nSEND THIS FILE:", BUNDLE["zip"], f"({BUNDLE['zip'].stat().st_size / 1024:.0f} KB)")

runs done: 42 | runs with a failed stage: 0


,rows
block_progress,930
clusters,1036
fidelity,3876
regimes,320
similarity_block,78348
tracking,3360



SEND THIS FILE: c:\Users\arthu\USPy\10_Mestrado\fedWater\notebooks\fedwater_lab_refactor\poc05c_similarity\POC_05c_similarity_bundle.zip (858 KB)


## 6. A first look (orientation only — the reading happens on the bundle)

Median per federated space × arm, pooled over runs. Q1 against the demand ceiling, Q2 in the
phases where the partition is non-trivial, Q3 as the share of drift pairs with the expected sign.

In [6]:
H = BUNDLE["headline"]
cols = [c for c in ("fid_dem_init", "fid_dem_transition", "fid_dem_final",
                    "ari_init", "ari_final", "top_ok_init", "top_ok_final",
                    "track_sign_ok", "track_rho_signed_median",
                    "track_rho_nondrift_abs_median") if c in H.columns]
if len(H) and cols:
    agg = H.groupby(["space", "arm"])[cols].median(numeric_only=True).round(3)
    agg["runs"] = H.groupby(["space", "arm"]).size()
    display(agg)
    if "world_hash" in H.columns:
        display(H.groupby(["world_hash", "space", "arm"])[[c for c in cols if c.startswith(("ari", "track_sign"))]]
                 .median(numeric_only=True).round(3))

fid_dem_init  fid_dem_transition  fid_dem_final  ari_init  ari_final  track_sign_ok  track_rho_signed_median  track_rho_nondrift_abs_median  runs
space      arm                                                                                                                                                      
s_dec      all            0.343               0.657          0.543       1.0      0.500          1.000                    0.646                          0.643    40
           mixed          0.343               0.686          0.514       1.0      0.000          1.000                    0.607                          0.706    40
           pure           0.457               0.686          0.600       1.0      1.000          1.000                    0.675                          0.572    40
s_dec_abs  all            0.486               0.714          0.657       1.0      1.000          1.000                    0.673                          0.665    40
           mixed          0.457               0.714          0.514       1.0      0.500          1.000                    0.624                          0.679    40
           pure           0.314               0.657          0.714       1.0      1.000          1.000                    0.685                          0.545    40
s_dem      demand           NaN                 NaN            NaN       1.0      1.000          1.000                    0.982                          0.127    40
s_lat      latent         0.057               0.400          0.229       1.0     -0.200          0.667                    0.778                          0.895    40
s_true     all            0.086               0.657          0.771       1.0      0.000          1.000                    0.691                          0.640    40
           mixed          0.029               0.486          0.257       1.0     -0.100          1.000                    0.682                          0.640    40
           pure           0.600               0.829          0.943       1.0      1.000          1.000                    0.713                          0.169    40
s_true_abs all              NaN                 NaN            NaN       1.0      1.000          1.000                    0.789                          0.670    40
           mixed            NaN                 NaN            NaN       1.0     -0.333          0.667                    0.848                          0.675    40
           pure             NaN                 NaN            NaN       1.0      1.000          1.000                    0.809                          0.169    40

ari_init  ari_final  track_sign_ok
world_hash   space      arm                                       
16ce72c589df s_dec      all          NaN        NaN          1.000
                        mixed        NaN        NaN          1.000
                        pure         NaN        NaN          1.000
             s_dec_abs  all          NaN        NaN          1.000
                        mixed        NaN        NaN          1.000
                        pure         NaN        NaN          1.000
             s_dem      demand       NaN        NaN          1.000
             s_lat      latent       NaN        NaN          1.000
             s_true     all          NaN        NaN          1.000
                        mixed        NaN        NaN          0.667
                        pure         NaN        NaN          1.000
             s_true_abs all          NaN        NaN          1.000
                        mixed        NaN        NaN          0.667
                        pure         NaN        NaN          1.000
ece8f4ad92e5 s_dec      all          NaN      0.500          1.000
                        mixed        NaN      0.500          1.000
                        pure         NaN      1.000          1.000
             s_dec_abs  all          NaN      1.000          1.000
                        mixed        NaN      1.000          1.000
                        pure         NaN      1.000          1.000
             s_dem      demand       NaN      1.000          1.000
             s_lat      latent       NaN      1.000          1.000
             s_true     all          NaN      1.000          1.000
                        mixed        NaN      0.000          1.000
                        pure         NaN      1.000          1.000
             s_true_abs all          NaN      1.000          1.000
                        mixed        NaN     -0.333          0.667
                        pure         NaN      1.000          1.000
f335b2cfad67 s_dec      all          1.0      0.400          1.000
                        mixed        1.0     -0.200          1.000
                        pure         1.0      0.400          1.000
             s_dec_abs  all          1.0      0.400          1.000
                        mixed        1.0     -0.200          1.000
                        pure         1.0     -0.200          1.000
             s_dem      demand       1.0      1.000          1.000
             s_lat      latent       1.0     -0.200          0.500
             s_true     all          1.0     -0.200          1.000
                        mixed        1.0     -0.200          1.000
                        pure         1.0      1.000          1.000
             s_true_abs all          1.0     -0.200          1.000
                        mixed        1.0     -0.200          1.000
                        pure         1.0      1.000          1.000